# Matching and comparison of the database with EM-DAT 

In [3]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [32]:
import pandas as pd
import json
from collections import Counter
import numpy as np
import pandas as pd
import seaborn as sns
import spacy
import re
import pycountry
import sys
import copy as cp
from random import randrange, randint
import ast
import geopy as gpy
import itertools
import time
from shapely.geometry import Polygon, Point

import geopandas as gpd

from src.text_processing_functions import *
from src.LLM_functions import *
from src.plot_functions import *
from src.data import *
from src.post_process_functions import *
from src.hazard_def import *

#sys.path.append('/home/lhasbini/como_school/como_project4/src/')

In [93]:
#Post process files 
df_llm = pd.read_csv(DATA_PATH + 'results_proc/'+'post_processed_llm_response_impact_labelled_reports_test_continue_16rep_meta-llama_llama-4-scout-17b-16e-instruct.csv', encoding='utf-8')
df_llm_geo = gpd.read_file(DATA_PATH + 'results_proc/'+'post_processed_llm_response_impact_labelled_reports_test_continue_16rep_meta-llama_llama-4-scout-17b-16e-instruct_geo.gpkg')

#Conver columns to list 
col_to_list = ['country', 'location', 'country_kw', 'hazards']
for col in col_to_list : 
        df_llm[col] = df_llm[col].apply(
            lambda x: ast.literal_eval(x) if pd.notna(x) and isinstance(x, str) and x.strip().startswith("[") else ([x] if pd.notna(x) else None)
        )
        df_llm_geo[col] = df_llm_geo[col].apply(
            lambda x: ast.literal_eval(x) if pd.notna(x) and isinstance(x, str) and x.strip().startswith("[") else ([x] if pd.notna(x) else None)
        )

In [ ]:
## Gather the impact at country level 


In [94]:
df_llm_geo

,impactSubtype,impactValue,impactUnit,impactValuePrecision,country,location,startYear,startMonth,startDay,endYear,...,unit_type,gaul0_code,gaul1_code,gaul2_code,locationLowestAdmin,geocoding_country_flag,geocoding_osm_flag,locationOsm,locationGaul,geometry
0,Residential Buildings,205368.00,homes,exact,[Bangladesh],[19 affected districts],2020.0,5.0,20.0,NaN,...,other,[229.0],[None],[None],ADM_0,1,0,['Bangladesh'],['Bangladesh'],"MULTIPOLYGON (((89.06422 21.82209, 89.06381 21..."
1,Residential Buildings,55767.00,homes,exact,[Bangladesh],[19 affected districts],2020.0,5.0,20.0,NaN,...,other,[229.0],[None],[None],ADM_0,1,0,['Bangladesh'],['Bangladesh'],"MULTIPOLYGON (((89.06422 21.82209, 89.06381 21..."
2,Crop Production and Forestry,320.37,km**2 of crop production and forestry,exact,[Bangladesh],[19 affected districts],2020.0,5.0,20.0,NaN,...,km**2,[229.0],[None],[None],ADM_0,1,0,['Bangladesh'],['Bangladesh'],"MULTIPOLYGON (((89.06422 21.82209, 89.06381 21..."
3,"Water, Sanitation, and Hygiene Infrastructure",18235.00,"water, sanitation and hygiene facilities",exact,[Bangladesh],[19 affected districts],2020.0,5.0,20.0,NaN,...,other,[229.0],[None],[None],ADM_0,1,0,['Bangladesh'],['Bangladesh'],"MULTIPOLYGON (((89.06422 21.82209, 89.06381 21..."
4,Other Transportation Infrastructure,440.00,km of roads,exact,[Bangladesh],[19 affected districts],2020.0,5.0,20.0,NaN,...,km,[229.0],[None],[None],ADM_0,1,0,['Bangladesh'],['Bangladesh'],"MULTIPOLYGON (((89.06422 21.82209, 89.06381 21..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
272,Access to Healthcare,NaN,people,None,[Zambia],[Zambia],NaN,NaN,NaN,NaN,...,other,[168.0],[None],[None],ADM_0,0,0,['Zambia'],['Zambia'],"MULTIPOLYGON (((32.92086 -9.4079, 32.92303 -9...."
273,IT and Communication Infrastructure,NaN,undefined IT and communication facilities,None,[Zambia],[Zambia],NaN,NaN,NaN,NaN,...,other,[168.0],[None],[None],ADM_0,0,0,['Zambia'],['Zambia'],"MULTIPOLYGON (((32.92086 -9.4079, 32.92303 -9...."
274,Other Economic and Livelihood Impacts,1525573.00,CHF,exact,[Zambia],[Zambia],NaN,NaN,NaN,NaN,...,other,[168.0],[None],[None],ADM_0,0,0,['Zambia'],['Zambia'],"MULTIPOLYGON (((32.92086 -9.4079, 32.92303 -9...."
275,Residential Buildings,NaN,houses,None,[Zambia],[Zambia],NaN,NaN,NaN,NaN,...,other,[168.0],[None],[None],ADM_0,0,0,['Zambia'],['Zambia'],"MULTIPOLYGON (((32.92086 -9.4079, 32.92303 -9...."


In [73]:
#Open EM-DAT 
df_em_dat = pd.read_excel(DATA_EXTERNAL_SOURCE + 'public_emdat_custom_request_2025-08-19.xlsx')
# df_em_dat = df_em_dat[["DisNo.", "Disaster Type", 'Country', 'Subregion', 'Region', 'Location', 
#                        'Start Year', 'Start Month', 'Start Day', 'End Year', 'End Month', 'End Day',
#                        'Total Deaths', 'No. Injured', 'No. Affected', 'No. Homeless',
#                        'Total Affected', "Reconstruction Costs ('000 US$)",
#                        "Reconstruction Costs, Adjusted ('000 US$)", "Insured Damage ('000 US$)", "Insured Damage, Adjusted ('000 US$)",
#                        "Total Damage ('000 US$)", "Total Damage, Adjusted ('000 US$)"]]

In [87]:
df_em_dat.columns

Index(['DisNo.', 'Historic', 'Classification Key', 'Disaster Group',
       'Disaster Subgroup', 'Disaster Type', 'Disaster Subtype',
       'External IDs', 'Event Name', 'ISO', 'country_emdat', 'Subregion',
       'Region', 'Location', 'Origin', 'Associated Types', 'OFDA/BHA Response',
       'Appeal', 'Declaration', 'AID Contribution ('000 US$)', 'Magnitude',
       'Magnitude Scale', 'Latitude', 'Longitude', 'River Basin', 'startYear',
       'Start Month', 'Start Day', 'End Year', 'End Month', 'End Day',
       'Total Deaths', 'No. Injured', 'No. Affected', 'No. Homeless',
       'Total Affected', 'Reconstruction Costs ('000 US$)',
       'Reconstruction Costs, Adjusted ('000 US$)',
       'Insured Damage ('000 US$)', 'Insured Damage, Adjusted ('000 US$)',
       'Total Damage ('000 US$)', 'Total Damage, Adjusted ('000 US$)', 'CPI',
       'Admin Units', 'Entry Date', 'Last Update'],
      dtype='object')

In [89]:
df_em_dat.iloc[0]

DisNo.                                                  1900-0003-USA
Historic                                                          Yes
Classification Key                                    nat-met-sto-tro
Disaster Group                                                Natural
Disaster Subgroup                                      Meteorological
Disaster Type                                                   Storm
Disaster Subtype                                     Tropical cyclone
External IDs                                                      NaN
Event Name                                                        NaN
ISO                                                               USA
country_emdat                                United States of America
Subregion                                            Northern America
Region                                                       Americas
Location                                            Galveston (Texas)
Origin              

In [75]:
# Map hazards to emdat hazards 
reverse_mapping = {}
for main, subs in hazard_mapping_emdat.items():
    for s in subs:
        reverse_mapping[s.lower()] = main   # lowercase for robustness

# --- For df_em_dat (single hazard per row) ---
df_em_dat["Disaster Type"] = df_em_dat["Disaster Type"].str.lower().map(reverse_mapping)

# --- For df_llm (list of hazards per row) ---
def map_hazard_list(hazard_list):
    mapped = []
    for h in hazard_list:
        h_low = h.lower()
        if h_low in reverse_mapping:
            mapped.append(reverse_mapping[h_low])
    return list(set(mapped))  # remove duplicates

df_llm["hazards"] = df_llm["hazards"].apply(map_hazard_list)

In [76]:
## Rename emdat columns 
df_em_dat = df_em_dat.rename({'Country' : 'country_emdat', 'Start Year' : 'startYear'}, axis=1)

# First merge on country + startYear
df_llm_em_dat = df_em_dat.merge(df_llm, on=['startYear'], how='inner')

# Keep only rows where Disaster Type is inside hazards list
df_llm_em_dat = df_llm_em_dat[
    df_llm_em_dat.apply(lambda row: row["Disaster Type"] in row["hazards"], axis=1)
].reset_index(drop=True)

df_llm_em_dat = df_llm_em_dat[
    df_llm_em_dat.apply(lambda row: row["country_emdat"] in row["country"], axis=1)
].reset_index(drop=True)

In [77]:
df_llm_em_dat.head()

,DisNo.,Historic,Classification Key,Disaster Group,Disaster Subgroup,Disaster Type,Disaster Subtype,External IDs,Event Name,ISO,...,reportLink,disasterType,nathaz_text,country_iso3,country_iso3_kw,impactSubtype_orig,hazards_reclass,impactValueOrig,impactUnitOrig,unit_type
0,2017-0224-MYS,No,nat-hyd-flo-flo,Natural,Hydrological,Flood,Flood (General),DFO:4443,NaN,MYS,...,https://adore.ifrc.org/Download.aspx?FileId=17...,Flood,['a. situation analysis description of the dis...,NaN,MYS,Displaced People,['Flood'],23000.0,people,other
1,2017-0224-MYS,No,nat-hyd-flo-flo,Natural,Hydrological,Flood,Flood (General),DFO:4443,NaN,MYS,...,https://adore.ifrc.org/Download.aspx?FileId=17...,Flood,['a. situation analysis description of the dis...,NaN,MYS,Human Deaths,['Flood'],1.0,person,other
2,2017-0224-MYS,No,nat-hyd-flo-flo,Natural,Hydrological,Flood,Flood (General),DFO:4443,NaN,MYS,...,https://adore.ifrc.org/Download.aspx?FileId=17...,Flood,['a. situation analysis description of the dis...,NaN,MYS,Human Health and Wellbeing,['Flood'],NaN,NaN,other
3,2017-0224-MYS,No,nat-hyd-flo-flo,Natural,Hydrological,Flood,Flood (General),DFO:4443,NaN,MYS,...,https://adore.ifrc.org/Download.aspx?FileId=17...,Flood,['a. situation analysis description of the dis...,NaN,MYS,Access to Healthcare,['Flood'],NaN,NaN,other
4,2017-0224-MYS,No,nat-hyd-flo-flo,Natural,Hydrological,Flood,Flood (General),DFO:4443,NaN,MYS,...,https://adore.ifrc.org/Download.aspx?FileId=17...,Flood,['a. situation analysis description of the dis...,NaN,MYS,"Water, Sanitation, and Hygiene Infrastructure",['Flood'],3000.0,hygiene kits,other


In [78]:
appeal_disno_counts = (
    df_llm_em_dat
    .groupby("appealCode")["DisNo."]
    .nunique()
    .reset_index(name="unique_DisNo_count")
)

In [79]:
appeal_disno_counts

,appealCode,unique_DisNo_count
0,MDRBD022,4
1,MDRBJ019,1
2,MDRCM039,2
3,MDRCN006,8
4,MDRIQ014,3
5,MDRKE058,4
6,MDRMY003,3
7,MDRNG041,2
8,MDRPK026,2
9,MDRRW022,1


In [80]:
DisNo_MDRBD022=df_llm_em_dat.loc[df_llm_em_dat.appealCode=="MDRCN006"]["DisNo."].unique()

In [81]:
df_llm.loc[df_llm.appealCode=="MDRCN006"]

,impactSubtype,impactValue,impactUnit,impactValuePrecision,country,location,startYear,startMonth,startDay,endYear,...,reportLink,disasterType,nathaz_text,country_iso3,country_iso3_kw,impactSubtype_orig,hazards_reclass,impactValueOrig,impactUnitOrig,unit_type
55,Affected People,1381000.0,people,exact,[China],"[Sichuan, Gansu]",2018.0,7.0,7.0,NaN,...,https://adore.ifrc.org/Download.aspx?FileId=23...,Flood,['dref operation operation n° mdrcn006 date of...,NaN,CHN,Affected People,['Flood'],1381000.0,people,other
56,Human Deaths,15.0,people,exact,[China],"[Sichuan, Gansu]",2018.0,7.0,7.0,NaN,...,https://adore.ifrc.org/Download.aspx?FileId=23...,Flood,['dref operation operation n° mdrcn006 date of...,NaN,CHN,Human Deaths,['Flood'],15.0,people,other
57,Displaced People,222000.0,people,exact,[China],[Sichuan],2018.0,7.0,7.0,NaN,...,https://adore.ifrc.org/Download.aspx?FileId=23...,Flood,['dref operation operation n° mdrcn006 date of...,NaN,CHN,Displaced People,['Flood'],222000.0,people,other
58,Missing People,4.0,people,exact,[China],[Gansu],2018.0,7.0,10.0,NaN,...,https://adore.ifrc.org/Download.aspx?FileId=23...,Flood,['dref operation operation n° mdrcn006 date of...,NaN,CHN,Missing People,['Flood'],4.0,people,other
59,Residential Buildings,900.0,homes,exact,[China],[Sichuan],2018.0,7.0,7.0,NaN,...,https://adore.ifrc.org/Download.aspx?FileId=23...,Flood,['dref operation operation n° mdrcn006 date of...,NaN,CHN,Residential Buildings,['Flood'],900.0,houses,other
60,Residential Buildings,2300.0,homes,exact,[China],[Gansu],2018.0,7.0,10.0,NaN,...,https://adore.ifrc.org/Download.aspx?FileId=23...,Flood,['dref operation operation n° mdrcn006 date of...,NaN,CHN,Residential Buildings,['Flood'],2300.0,houses,other
61,Other Economic and Livelihood Impacts,792000000.0,CHF,exact,[China],[Sichuan],2018.0,7.0,7.0,NaN,...,https://adore.ifrc.org/Download.aspx?FileId=23...,Flood,['dref operation operation n° mdrcn006 date of...,NaN,CHN,Economy and Market,['Flood'],792000000.0,CHF,other
62,Other Economic and Livelihood Impacts,538000000.0,CHF,exact,[China],[Gansu],2018.0,7.0,10.0,NaN,...,https://adore.ifrc.org/Download.aspx?FileId=23...,Flood,['dref operation operation n° mdrcn006 date of...,NaN,CHN,Economy and Market,['Flood'],538000000.0,CHF,other
63,Crop Production and Forestry,369.0,km**2 of undefined crop production and forestry,exact,[China],[Sichuan],2018.0,7.0,7.0,NaN,...,https://adore.ifrc.org/Download.aspx?FileId=23...,Flood,['dref operation operation n° mdrcn006 date of...,NaN,CHN,Crop Production and Forestry,['Flood'],36900.0,hectares,km**2
64,Human Health and Wellbeing,NaN,unknown,NaN,[China],"[Sichuan, Gansu]",2018.0,7.0,7.0,NaN,...,https://adore.ifrc.org/Download.aspx?FileId=23...,Flood,['dref operation operation n° mdrcn006 date of...,NaN,CHN,Human Health and Wellbeing,['Flood'],NaN,NaN,other


In [91]:
df_em_dat.loc[df_em_dat['DisNo.'].isin(DisNo_MDRBD022)][["DisNo.", "Disaster Type", 'country_emdat', 'Subregion', 'Region', 'Location', 
                       'startYear', 'Start Month', 'Start Day', 'End Year', 'End Month', 'End Day',
                       'Total Deaths', 'No. Injured', 'No. Affected', 'No. Homeless',
                       'Total Affected', 'Admin Units', 'Entry Date', 'Last Update']]

,DisNo.,Disaster Type,country_emdat,Subregion,Region,Location,startYear,Start Month,Start Day,End Year,End Month,End Day,Total Deaths,No. Injured,No. Affected,No. Homeless,Total Affected,Admin Units,Entry Date,Last Update
14457,2018-0159-CHN,Flood,China,Eastern Asia,Asia,Guangxi Zhuang Autonomous,2018,5.0,7.0,2018,5.0,16.0,5.0,NaN,70000.0,NaN,70000.0,"[{""adm1_code"":904,""adm1_name"":""Guangxi Zhuangz...",2018-05-23,2023-09-25
14479,2018-0198-CHN,Flood,China,Eastern Asia,Asia,"Sichuan, Gansu, Chongqing, Hubei, Jiangsu, Gui...",2018,5.0,5.0,2018,7.0,31.0,112.0,NaN,450000.0,NaN,450000.0,"[{""adm1_code"":898,""adm1_name"":""Anhui Sheng""},{...",2018-07-19,2023-09-25
14492,2018-0210-CHN,Flood,China,Eastern Asia,Asia,"Jiangxi, Hebei, Shanxi, Jiangsu, Shandong, Hen...",2018,5.0,7.0,2018,5.0,30.0,77.0,NaN,225000.0,NaN,225000.0,"[{""adm1_code"":900,""adm1_name"":""Chongqing Shi""}...",2018-07-20,2023-09-25
14493,2018-0211-CHN,Flood,China,Eastern Asia,Asia,"Fujian, Guangdong, Guangxi",2018,5.0,7.0,2018,5.0,14.0,2.0,NaN,6000.0,NaN,6000.0,"[{""adm1_code"":901,""adm1_name"":""Fujian Sheng""},...",2018-07-20,2023-09-25
14531,2018-0250-CHN,Flood,China,Eastern Asia,Asia,"Deyang, Mianyang, Guangyuan prefectures (Sichu...",2018,7.0,7.0,2018,7.0,7.0,3.0,NaN,1381000.0,NaN,1381000.0,"[{""adm2_code"":13259,""adm2_name"":""Deyang""},{""ad...",2018-07-27,2023-09-25
14532,2018-0252-CHN,Flood,China,Eastern Asia,Asia,"Tianshui, Zhangye, Pingliang (Gansu province)",2018,7.0,10.0,2018,7.0,11.0,16.0,NaN,1519000.0,NaN,1519000.0,"[{""adm2_code"":13018,""adm2_name"":""Tianshui""},{""...",2018-07-27,2023-09-25
14548,2018-0282-CHN,Flood,China,Eastern Asia,Asia,NaN,2018,6.0,28.0,2018,7.0,5.0,11.0,NaN,36000.0,NaN,36000.0,NaN,2018-08-09,2023-09-25
14597,2018-0369-CHN,Flood,China,Eastern Asia,Asia,NaN,2018,8.0,29.0,2018,9.0,5.0,18.0,NaN,11400.0,NaN,11400.0,NaN,2018-10-11,2023-09-25


In [ ]:
df_em_dat.loc[df_em_dat['DisNo.'].isin(DisNo_MDRBD022)][["DisNo.", "Disaster Type", 'country_emdat', 'Subregion', 'Region', 'Location', 
                       'startYear', 'Start Month', 'Start Day', 'End Year', 'End Month', 'End Day',
                       'Total Deaths', 'No. Injured', 'No. Affected', 'No. Homeless',
                       'Total Affected', 'Admin Units', 'Entry Date', 'Last Update']]